#### Importing required libraries 

In [19]:
from utils import Load_Rumours_Dataset_filtering_since_first_post
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import *
import pandas as pd
import time
import optuna
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")
event_name ="ferguson"

In [5]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


In [6]:
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=40*60*24)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()
test.shape

(352, 10)

In [4]:
previous_node_count = 0

In [5]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 258


#### Tunning Random Forest

In [124]:


n_train = len(X_train)  
def objective_rf(trial, X, y):
    # sensible RF hyperparameter search space
    param_grid = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 5),                     # tree depth
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),     # internal node split
        # min_samples_leaf relative to dataset size but never larger than n_train:
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            max(2, min(n_train, int(max(2, n_train * 0.2))))
        ),
        # max_features: either a rule or a fraction; use categorical choices for stability
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        # optional class weight (useful for imbalance)
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
    }

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=1337)
    cv_scores = np.empty(cv.get_n_splits())

    for idx, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = RandomForestClassifier(
            n_estimators=param_grid["n_estimators"],
            max_depth=param_grid["max_depth"],
            min_samples_split=param_grid["min_samples_split"],
            min_samples_leaf=param_grid["min_samples_leaf"],
            max_features=param_grid["max_features"],
            bootstrap=param_grid["bootstrap"],
            class_weight=param_grid["class_weight"],
            n_jobs=-1,
            random_state=1337,
        )

        model.fit(X_tr, y_tr)

        # Predict probabilities on the validation fold (choose threshold on validation)
        if hasattr(model, "predict_proba"):
            y_val_prob = model.predict_proba(X_val)[:, 1]
        else:
            # fallback (shouldn't happen for RF)
            y_val_prob = model.predict(X_val)

        thresholds = np.linspace(0.01, 0.99, 99)
        f1_scores = [f1_score(y_val, (y_val_prob > t).astype(int)) for t in thresholds]
        best_idx = int(np.nanargmax(f1_scores))
        best_threshold = thresholds[best_idx]

        y_val_pred = (y_val_prob > best_threshold).astype(int)
        f1_val = f1_score(y_val, y_val_pred)

        # report intermediate value to Optuna and allow pruning
        trial.report(f1_val, step=idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

        cv_scores[idx] = f1_val

    return float(np.mean(cv_scores))


In [ ]:
# Run study
start_time = time.time()

study = optuna.create_study(direction="maximize", study_name="Random Forest Charlie Hebdo",
                            sampler=optuna.samplers.TPESampler(seed=111113857),
                            pruner=optuna.pruners.MedianPruner())
func = lambda trial: objective_rf(trial, pd.DataFrame(X_train), y_train.astype(int))
study.optimize(func, n_trials=100)

end_time = time.time()

hyper_tuning_time = end_time - start_time

print(f"\tBest value (bcr1p_sum): {study.best_value:.5f}")
print(f"\tBest params:")

for key, value in study.best_params.items():
    print(f"\t\t{key}: {value}")


In [ ]:
best_params = study.best_params

In [12]:
best_params= {'n_estimators': 250,
 'max_depth': 4,
 'min_samples_split': 47,
 'min_samples_leaf': 64,
 'max_features': 'log2'}

#### Example  training

In [10]:

model = RandomForestClassifier(
   **best_params,
    n_jobs=-1,
    random_state=42,
    verbose=False
)
# Train the model
model.fit(
    X_train,
    y_train
)


RandomForestClassifier(class_weight='balanced_subsample', max_depth=4,
                       max_features='log2', min_samples_leaf=64,
                       min_samples_split=47, n_estimators=250, n_jobs=-1,
                       random_state=42, verbose=False)

In [11]:
y_train_prob = model.predict_proba(X_train)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_train_pred = (y_train_prob > best_threshold).astype(int)
y_test_pred = (y_test_prob > best_threshold).astype(int)

# Evaluation function
def evaluate(y_true, y_pred, y_prob, label=""):
    print(f"  - Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  - Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  - Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  - AUC:       {roc_auc_score(y_true, y_prob):.4f}")
    print("")

# Show metrics
print('Train Set: ')
evaluate(y_train, y_train_pred, y_train_prob, label="Train")
print('Test Set: ')
evaluate(y_test, y_test_pred, y_test_prob, label="Test")

Train Set: 
  - Accuracy:  0.8500
  - Precision: 0.8028
  - Recall:    0.9044
  - AUC:       0.9313

Test Set: 
  - Accuracy:  0.6733
  - Precision: 0.4929
  - Recall:    0.9286
  - AUC:       0.7977



#### Setting MLflow Experiment

In [20]:
event_name ='ferguson'

In [21]:
from datetime import date
import mlflow

today = date.today()
formatted_today = today.strftime("%Y-%m-%d")


mlflow.set_experiment(f"Random Forest {formatted_today} {event_name} run")

2026/07/05 18:30:18 INFO mlflow.tracking.fluent: Experiment with name 'Random Forest 2026-07-05 ferguson run' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/mlruns/126', creation_time=1783276218923, experiment_id='126', last_update_time=1783276218923, lifecycle_stage='active', name='Random Forest 2026-07-05 ferguson run', tags={}, workspace='default'>

#### Loading dataset statistics to get the final time cut 

In [22]:
import pandas as pd
df_posts_by_time_cut = pd.read_pickle(f'replies_{event_name}.pkl')

df_posts_by_time_cut['min_since_fst_post'] = round(
            (df_posts_by_time_cut['time'] - df_posts_by_time_cut['time'].min()).dt.total_seconds() / 60, 2)


In [23]:
df_metrics = df_posts_by_time_cut[['id','time','rumour','min_since_fst_post']].drop_duplicates().sort_values(by='time')

In [24]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=10000)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [25]:

start = test.min_since_fst_post.min()
end = test.min_since_fst_post.max()
duration = end-start
experiment_time= duration+60



In [13]:
print('start: ',start)
print('end: ',end)
print('experiment_time: ',experiment_time)

start:  707.38
end:  1016.1
experiment_time:  368.72


In [14]:
def compute_metrics_custom(df, prob_col='prob', target_col='rumour'):
    df = df.copy()
    df = df.sort_values(prob_col, ascending=False).reset_index(drop=True)

    total_frauds = df[target_col].sum()
    total_records = df.shape[0]
    n = len(df)

    # ✅ Bins from 1% to 100% in 1% steps
    bins = np.arange(0.01, 1.01, 0.01)

    results = []

    for p in bins:
        cutoff = int(np.ceil(n * p))
        subset = df.iloc[:cutoff]

        frauds = subset[target_col].sum()
        records = len(subset)

        results.append({
            'percentile': round(p * 100, 0),
            'records': records,
            'frauds_captured': frauds,
            'capture_rate': frauds / total_frauds if total_frauds > 0 else 0,
            'bad_rate': frauds / records if records > 0 else 0,
            'false_positive_rate': (records - frauds) / total_records if records > 0 else 0
        })

    df_out = (
        pd.DataFrame(results)
        .drop_duplicates(subset='percentile')
        .sort_values('percentile')
        .reset_index(drop=True)
    )

    return df_out

In [13]:
import mlflow
import mlflow.lightgbm
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
import lightgbm as lgb

previous_node_count = 0

for time_cut in np.linspace(0, int(experiment_time), 50):
    print(f"\n=== Time Cut: {time_cut} minutes ===")
    
    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train, test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    model = RandomForestClassifier(
        n_estimators=500,              # enough trees for stability
        max_depth=6,                  # CRITICAL: limit depth (prevents overfitting)
        min_samples_leaf=100,         # strong regularization
        min_samples_split=200,        # avoid tiny splits
    
        max_features="sqrt",          # randomness + reduces dominance of embedding
        bootstrap=True,
    
        class_weight="balanced",      # handles 15% bad rate
    
        n_jobs=-1,
        random_state=42
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        current_df_metrics = df_metrics.iloc[X_train.shape[0]:X_train.shape[0]+X_test.shape[0]]
        current_df_metrics['prob'] = y_test_prob
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]
 

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])
        df_metrics_by_bucket.to_csv(f"metrics_by_bucket_{event_name}_RF.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_{event_name}_RF.csv")

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 0.0 minutes ===
New Instances: 1

=== Time Cut: 41.775510204081634 minutes ===
New Instances: 10

=== Time Cut: 83.55102040816327 minutes ===
New Instances: 4

=== Time Cut: 125.32653061224491 minutes ===
New Instances: 4

=== Time Cut: 167.10204081632654 minutes ===
New Instances: 3

=== Time Cut: 208.87755102040816 minutes ===
New Instances: 4

=== Time Cut: 250.65306122448982 minutes ===
New Instances: 1

=== Time Cut: 292.42857142857144 minutes ===
New Instances: 0

=== Time Cut: 334.2040816326531 minutes ===
New Instances: 4

=== Time Cut: 375.9795918367347 minutes ===
New Instances: 0

=== Time Cut: 417.7551020408163 minutes ===
New Instances: 0

=== Time Cut: 459.53061224489795 minutes ===
New Instances: 0

=== Time Cut: 501.30612244897964 minutes ===
New Instances: 0

=== Time Cut: 543.0816326530612 minutes ===
New Instances: 0

=== Time Cut: 584.8571428571429 minutes ===
New Instances: 0

=== Time Cut: 626.6326530612245 minutes ===
New Instances: 0

=== Time Cut

In [26]:
new_posts_times = np.sort(
    np.unique(
        np.ceil(
            df_metrics[
                (df_metrics.min_since_fst_post >= start ) &
                (df_metrics.min_since_fst_post <= end)
            ].min_since_fst_post - start
        )
    )
)

In [27]:
new_posts_times=[c for c in new_posts_times if c >10]

In [28]:
new_posts_times

[np.float64(11.0),
 np.float64(14.0),
 np.float64(15.0),
 np.float64(16.0),
 np.float64(17.0),
 np.float64(18.0),
 np.float64(19.0),
 np.float64(21.0),
 np.float64(22.0),
 np.float64(23.0),
 np.float64(24.0),
 np.float64(25.0),
 np.float64(26.0),
 np.float64(29.0),
 np.float64(32.0),
 np.float64(35.0),
 np.float64(37.0),
 np.float64(39.0),
 np.float64(42.0),
 np.float64(43.0),
 np.float64(44.0),
 np.float64(51.0),
 np.float64(55.0),
 np.float64(56.0),
 np.float64(57.0),
 np.float64(58.0),
 np.float64(60.0),
 np.float64(61.0),
 np.float64(63.0),
 np.float64(64.0),
 np.float64(66.0),
 np.float64(68.0),
 np.float64(69.0),
 np.float64(76.0),
 np.float64(79.0),
 np.float64(84.0),
 np.float64(87.0),
 np.float64(88.0),
 np.float64(90.0),
 np.float64(91.0),
 np.float64(93.0),
 np.float64(94.0),
 np.float64(99.0),
 np.float64(101.0),
 np.float64(102.0),
 np.float64(104.0),
 np.float64(106.0),
 np.float64(107.0),
 np.float64(108.0),
 np.float64(109.0),
 np.float64(110.0),
 np.float64(111.0),
 np

In [ ]:
##### import mlflow
import mlflow.lightgbm
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
import lightgbm as lgb

previous_node_count = 0

for time_cut in new_posts_times:
    print(f"\n=== Time Cut: {time_cut} minutes ===")
    
    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train, test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    model = RandomForestClassifier(
        n_estimators=500,              # enough trees for stability
        max_depth=6,                  # CRITICAL: limit depth (prevents overfitting)
        min_samples_leaf=100,         # strong regularization
        min_samples_split=200,        # avoid tiny splits
    
        max_features="sqrt",          # randomness + reduces dominance of embedding
        bootstrap=True,
    
        class_weight="balanced",      # handles 15% bad rate
    
        n_jobs=-1,
        random_state=42
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        current_df_metrics = df_metrics.iloc[X_train.shape[0]:X_train.shape[0]+X_test.shape[0]]
        current_df_metrics['prob'] = y_test_prob
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]
            new_df_metrics = current_df_metrics.iloc[-X_test_new.shape[0]:]
            new_df_metrics['prob'] = y_test_new_prob
            new_df_metrics_by_bucket = compute_metrics_custom(new_df_metrics)
 

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])
        new_df_metrics_by_bucket.to_csv(f"metrics_by_bucket_new_posts_{event_name}_RF.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_new_posts_{event_name}_RF.csv")

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 11.0 minutes ===
New Instances: 7

=== Time Cut: 14.0 minutes ===
New Instances: 4

=== Time Cut: 15.0 minutes ===
New Instances: 1

=== Time Cut: 16.0 minutes ===
New Instances: 0

=== Time Cut: 17.0 minutes ===
New Instances: 1

=== Time Cut: 18.0 minutes ===
New Instances: 2

=== Time Cut: 19.0 minutes ===
New Instances: 0

=== Time Cut: 21.0 minutes ===
New Instances: 2

=== Time Cut: 22.0 minutes ===
New Instances: 0

=== Time Cut: 23.0 minutes ===
New Instances: 1

=== Time Cut: 24.0 minutes ===
New Instances: 1

=== Time Cut: 25.0 minutes ===
New Instances: 1

=== Time Cut: 26.0 minutes ===
New Instances: 3

=== Time Cut: 29.0 minutes ===
New Instances: 3

=== Time Cut: 32.0 minutes ===
New Instances: 0

=== Time Cut: 35.0 minutes ===
New Instances: 2

=== Time Cut: 37.0 minutes ===
New Instances: 1

=== Time Cut: 39.0 minutes ===
New Instances: 1

=== Time Cut: 42.0 minutes ===
New Instances: 1

=== Time Cut: 43.0 minutes ===
New Instances: 1

=== Time Cut: 44.0 